# Timefolio Engine — Quickstart

This notebook walks through the most common use-cases of the `timefolio` library.

**Prerequisites**

```bash
# Install the library from the project root
pip install -e .            # core (requests only)
```

**Credentials**

Create a `.env` file in the project root (never commit this file):

```
TIMEFOLIO_EMAIL=you@example.com
TIMEFOLIO_PASSWORD=your_password
TIMEFOLIO_PF_ID=18762
```

Or set the environment variables directly before running this notebook.

## 0. Setup

In [1]:
import logging
import os

# Optional: load credentials from a .env file in the project root.
# Remove this block if you prefer to set env vars another way.
try:
    from dotenv import load_dotenv
    load_dotenv()  # reads ../.env relative to examples/
    load_dotenv(dotenv_path="../.env", override=False)
except ImportError:
    pass  # python-dotenv is optional; credentials can come from the shell env

# Show INFO-level log messages from the library so you can follow what's happening.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

In [2]:
from timefolio import TimefolioAPIClient
from timefolio import TimefolioTrader

## 1. Authentication

`TimefolioAPIClient` manages a single `requests.Session`.  
Call `login()` once — the Bearer token is automatically attached to every
subsequent request.

In [3]:
EMAIL    = os.environ["TIMEFOLIO_EMAIL"]
PASSWORD = os.environ["TIMEFOLIO_PASSWORD"]
PF_ID    = int(os.environ.get("TIMEFOLIO_PF_ID", 18762))

api = TimefolioAPIClient(email=EMAIL, password=PASSWORD)

if not api.login():
    raise RuntimeError("Login failed — check your credentials in .env")

print(f"Logged in.  Active portfolio ID: {PF_ID}")

2026-03-17 01:35:10,103 [INFO] timefolio.api_client: Attempting to log in as cmschs03@naver.com ...
2026-03-17 01:35:10,428 [INFO] timefolio.api_client: Login successful.


Logged in.  Active portfolio ID: 18762


## 2. Creating a Trader

`TimefolioTrader` wraps the authenticated client with trading-specific methods.

In [4]:
trader = TimefolioTrader(api_client=api, pf_id=PF_ID)
print(f"Trader ready (pfId={trader.pf_id})")

Trader ready (pfId=18762)


## 3. Switch Tournament (optional)

If you are enrolled in multiple contests, use `set_tournament()` to switch
the active portfolio by matching a substring of the contest name.

In [10]:
# Example — change "연습용 대회" to the actual contest name you want to trade in.
# trader.set_tournament(target_name="연습용 대회")
# print(f"Now using pfId={trader.pf_id}")

## 4. 실시간 연결 & 포트폴리오 조회

`connect_realtime()`은 SignalR WebSocket으로 서버에 연결해 보유잔고·주문·포지션 데이터를 실시간으로 수신합니다.  
주문 취소 헬퍼(`cancel_all_pending`, `cancel_pending_by_stock`)도 이 데이터를 사용합니다.

| 메서드 | 설명 |
|---|---|
| `connect_realtime()` | SignalR 연결 시작 |
| `get_holdings()` | 보유잔고 (NAV, 평가금액 등) |
| `get_orders()` | 전체 주문 목록 |
| `get_positions()` | 보유 포지션 목록 |
| `get_pending_orders()` | 미접수 주문만 필터 (`state == "Generated"`) |
| `get_error_orders()` | 에러 주문만 필터 |
| `disconnect_realtime()` | SignalR 연결 종료 |

In [ ]:
import time

trader.connect_realtime()
time.sleep(3)  # 데이터 수신 대기

print("보유잔고 :", trader.get_holdings())
print("주문목록 :", trader.get_orders())
print("포지션   :", trader.get_positions())
print("미접수   :", trader.get_pending_orders())
print("에러주문 :", trader.get_error_orders())

trader.disconnect_realtime()

## 5. Market Order (Immediate Execution)

Omit `hm0` / `hm1` to execute at the current market price right now.

| Parameter | Description |
|---|---|
| `prod_id` | Ticker with `A` prefix (e.g. `A005930` = Samsung Electronics) |
| `weight` | Fraction of total NAV — `0.05` = 5 % |
| `ls` | `"L"` Buy / `"S"` Sell |
| `limit_idx` | Order-book aggressiveness 1–10 (default 5) |

In [ ]:
# Buy Samsung Electronics (A005930) at 5 % portfolio weight, execute immediately.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
)
print(result)

2026-03-17 00:39:50,480 [INFO] timefolio.trader: Order submitted [L] A005930 weight=5.0% | limitPrc=None stopPrc=None hm0=00:39 hm1=None


{'data': '', 'warnings': []}


## 6. Scheduled / TWAP Order

Set `hm0` (start time) and `hm1` (end time) to enable **TWAP** (Time-Weighted
Average Price) execution.  The server slices the order across the given window.

You can also supply `target_date` to schedule on a future business day.

In [ ]:
# Buy Hyundai Motor (A005380) at 10 % weight.
# Execution is spread from 09:00 to 12:20 on the next trading day.
result = trader.order(
    prod_id="A001500",
    weight=0.10,
    ls="L",
    hm0="09:00",
    hm1="12:20",
    target_date="2026-03-17",  # must be a business day
)
print(result)

2026-03-17 00:39:50,525 [INFO] timefolio.trader: Order submitted [L] A001500 weight=10.0% | limitPrc=None stopPrc=None hm0=09:00 hm1=12:20


{'data': '', 'warnings': []}


## 7. Limit Order

Pass `limit_prc` to peg the order to a specific price.  
The server will not execute above (buy) or below (sell) this price.

In [ ]:
# Buy Samsung Electronics at a limit price of 55,000 KRW.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
    limit_prc=55_000,
)
print(result)

2026-03-17 00:39:50,568 [INFO] timefolio.trader: Order submitted [L] A005930 weight=5.0% | limitPrc=55000 stopPrc=None hm0=00:39 hm1=None


{'data': '', 'warnings': []}


## 8. Stop Order

Pass `stop_prc` to set a stop-loss or stop-breakout trigger.  
The order activates only when the market price crosses `stop_prc`.

In [ ]:
# Sell Samsung Electronics if the price drops to 50,000 KRW (stop-loss).
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="S",
    stop_prc=50_000,
)
print(result)

2026-03-17 00:39:50,609 [ERROR] timefolio.trader: Order submission failed (HTTP 400): {"errors":["매수도 반대 방향 미완료 주문 존재"]}


None


## 9. Limit + Stop (Bracket Order)

Combine both parameters for a bracket: the order only activates after
`stop_prc` is touched and is then capped by `limit_prc`.

In [ ]:
# Enter a long position in KOSPI ETF (A069500) only if price breaks above
# 32,000, and cap the buy price at 32,500.
result = trader.order(
    prod_id="A069500",
    weight=0.08,
    ls="L",
    limit_prc=32_500,
    stop_prc=32_000,
    hm0="09:00",
    hm1="11:30",
)
print(result)

2026-03-17 00:39:50,645 [ERROR] timefolio.trader: Order submission failed (HTTP 400): {"errors":["보통주 외 거래불가"]}


None


## 10. 주문 취소

| 메서드 | 설명 |
|---|---|
| `cancel_order(ord_id)` | 특정 주문 ID 취소 |
| `cancel_all_pending()` | 미접수 주문 전체 취소 |
| `cancel_pending_by_stock(prod_id)` | 특정 종목의 미접수 주문만 취소 |

> `ord_id`는 `get_orders()` 응답의 `"Id"` 필드 값입니다.  
> `cancel_all_pending()` / `cancel_pending_by_stock()`는 `connect_realtime()` 이후에 사용 가능합니다.

In [ ]:
# 특정 주문 ID 취소 — get_orders()의 "Id" 필드 값을 사용
cancel_result = trader.cancel_order(ord_id=1219617)
print(cancel_result)

In [ ]:
# 미접수 주문 전체 취소 (connect_realtime() 이후 사용 가능)
# trader.connect_realtime()
# time.sleep(3)
# cancelled = trader.cancel_all_pending()
# print(f"취소된 주문 수: {cancelled}")

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# Portfolio/Orders — 당일 주문 목록 조회
res = api.get("Portfolio/Orders", params={"pfId": PF_ID, "d": today})
print(res.status_code)
# res.json()  # 전체 응답 페이로드

## 11. Raw API Access

`TimefolioAPIClient` exposes `get()` and `post()` helpers for any endpoint
not yet wrapped by `TimefolioTrader`.

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# Example: raw GET to fetch portfolio details
res = api.get("Portfolio/Summary", params={"pfId": PF_ID, "d": today})
print(res.status_code)
# res.json()  # full response payload

404


---

## Parameter Reference

| Parameter | Type | Default | Description |
|---|---|---|---|
| `prod_id` | `str` | — | Ticker symbol, e.g. `"A005930"` |
| `weight` | `float` | — | Portfolio weight, `0.05` = 5 % |
| `ls` | `str` | `"L"` | `"L"` Long/Buy · `"S"` Short/Sell |
| `ex` | `str` | `"E"` | Execution algorithm type |
| `limit_idx` | `int` | `5` | Order-book depth aggressiveness (1–10) |
| `limit_prc` | `float\|None` | `None` | Exact limit price; `None` = algo/market |
| `stop_prc` | `float\|None` | `None` | Stop trigger price; `None` = disabled |
| `hm0` | `str\|None` | `None` | Start time `"HH:MM"`; `None` = immediate |
| `hm1` | `str\|None` | `None` | End time `"HH:MM"` for TWAP window |
| `target_date` | `str\|None` | `None` | Business date `"YYYY-MM-DD"`; `None` = today |